# PlacementIQ Data Preprocessing Pipeline (Universal Version)
This notebook implements the data preprocessing pipeline. It automatically detects if you are running it locally or on Google Colab, and adjusts file paths accordingly!

In [1]:
# Step 1: Mount Google Drive (if in Colab) and Load Data
import pandas as pd
import numpy as np
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Update these paths to where you uploaded the dataset in your Google Drive
    train_path = '/content/drive/MyDrive/PlacementIQ/dataset/train.csv'
    test_path = '/content/drive/MyDrive/PlacementIQ/dataset/test.csv'
    print("Running in Google Colab environment...")
except ModuleNotFoundError:
    # Running locally
    train_path = 'dataset/train.csv'
    test_path = 'dataset/test.csv'
    print("Running in Local environment...")

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df = pd.concat([df_train, df_test], axis=0, ignore_index=True)

print("Combined Dataset Shape:", df.shape)
display(df.head())

Running in Local environment...
Combined Dataset Shape: (50000, 15)


,Student_ID,Age,Gender,Degree,Branch,CGPA,Internships,Projects,Coding_Skills,Communication_Skills,Aptitude_Test_Score,Soft_Skills_Rating,Certifications,Backlogs,Placement_Status
0,45476,20,Male,B.Sc,ECE,8.75,1,5,8,5,78,5,2,0,Placed
1,29402,24,Female,B.Tech,ME,6.80,0,3,4,6,76,5,1,0,Not Placed
2,29197,19,Male,BCA,IT,6.32,1,3,5,8,66,7,2,1,Not Placed
3,31177,20,Female,B.Sc,Civil,4.50,0,1,1,4,57,6,0,1,Not Placed
4,33909,21,Female,MCA,IT,5.89,1,3,4,6,54,5,1,2,Not Placed


## Step 2: Duplicate Removal
Remove any duplicate rows in the dataset.

In [2]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows. New shape: {df.shape}")

Removed 0 duplicate rows. New shape: (50000, 15)


## Step 3: Missing Value Handling
Impute numerical columns with median and categorical columns with mode.

In [3]:
# Detect numerical and categorical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

if 'Student_ID' in numerical_cols:
    numerical_cols.remove('Student_ID')
    
target_col = 'Placement_Status'
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

print("Numerical Columns:", numerical_cols)
print("Categorical Columns:", categorical_cols)

# Impute Numerical
for col in numerical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Impute Categorical
for col in categorical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])
        
print("Remaining Missing Values:", df.isnull().sum().sum())

Numerical Columns: ['Age', 'CGPA', 'Internships', 'Projects', 'Coding_Skills', 'Communication_Skills', 'Aptitude_Test_Score', 'Soft_Skills_Rating', 'Certifications', 'Backlogs']
Categorical Columns: ['Gender', 'Degree', 'Branch']
Remaining Missing Values: 0


## Step 4: Target Encoding
Encode the target column (Placement_Status).

In [4]:
if df[target_col].dtype == object or str(df[target_col].dtype) in ["string", "category", "str"]:
    unique_vals = set(df[target_col].dropna().unique())
    if unique_vals == {"Placed", "Not Placed"}:
        df[target_col] = df[target_col].map({"Placed": 1, "Not Placed": 0})
        print("Mapped 'Placed'->1, 'Not Placed'->0")
    else:
        from sklearn.preprocessing import LabelEncoder
        le = LabelEncoder()
        df[target_col] = le.fit_transform(df[target_col].astype(str))
        print("Target Encoded classes:", le.classes_)

Mapped 'Placed'->1, 'Not Placed'->0


## Step 5: Categorical Feature Encoding
Encode categorical features using LabelEncoder.

In [5]:
from sklearn.preprocessing import LabelEncoder

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"Encoded {col} - classes: {le.classes_}")

all_numerical = numerical_cols + categorical_cols

Encoded Gender - classes: ['Female' 'Male']
Encoded Degree - classes: ['B.Sc' 'B.Tech' 'BCA' 'MCA']
Encoded Branch - classes: ['CSE' 'Civil' 'ECE' 'IT' 'ME']


## Step 6: Outlier Detection and Capping (IQR Method)

In [6]:
total_capped = 0
for col in all_numerical:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    if outliers > 0:
        df[col] = df[col].clip(lower=lower, upper=upper)
        total_capped += outliers
        
print(f"Total outliers capped: {total_capped}")

Total outliers capped: 8131


## Step 7: Feature Scaling
Standardize numerical features.

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[all_numerical] = scaler.fit_transform(df[all_numerical])
print("StandardScaler applied to features.")

StandardScaler applied to features.


## Step 8: Train/Test Split and Save

In [8]:
from sklearn.model_selection import train_test_split
import joblib

# Prepare X and y
feature_cols = [c for c in df.columns if c != target_col and c != "Student_ID"]
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set: {len(X_train)}, Test set: {len(X_test)}")

# Determine save locations based on environment
try:
    import google.colab
    output_dir = '/content/drive/MyDrive/PlacementIQ/outputs'
    model_dir = '/content/drive/MyDrive/PlacementIQ/models'
except ModuleNotFoundError:
    output_dir = 'outputs'
    model_dir = 'models'

os.makedirs(output_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

df.to_csv(os.path.join(output_dir, "cleaned_data.csv"), index=False)
joblib.dump(scaler, os.path.join(model_dir, "scaler.joblib"))
joblib.dump(encoders, os.path.join(model_dir, "encoders.joblib"))

print(f"Cleaned data and transformers saved successfully to {output_dir} and {model_dir}!")

Train set: 40000, Test set: 10000
Cleaned data and transformers saved successfully to outputs and models!
